# **Image alignment**

<div style="color:#777777;margin-top: -15px;">
<b>Author</b>: Norman Juchler |
<b>Course</b>: ADLS ISP |
<b>Version</b>: v1.2 <br><br>
<!-- Date: 06.05.2025 -->
<!-- Comments: Entirely refactored -->
</div>

In a previous notebook, we explored how to move, rotate, scale, and otherwise transform an image. In this tutorial, you will learn how to align two images automatically.

**Goal**: Insert the missing puzzle piece  
**Input**: 2D image (see below)  
**Output**: Transformed 2D image  
**Methods**: Translate, rotate, scale, warp (using OpenCV)  
**Assumption**: Perspective distortion can be ignored


To "insert the missing piece," we need to find the *affine transformation* that aligns the puzzle piece (`imageB`) with the larger arrangement (imageA). In other words, our goal is to compute a suitable transformation matrix $M$.


### **Input**

<img src="../data/images/puzzle-input.jpg" alt="Puzzle problem" width="70%" />


### **Output**

<img src="../data/images/puzzle-output.jpg" alt="Puzzle problem solved" width="70%" />

 

---


## **☆ Exercise 1: Develop an alignment strategy**

You have learned already how to align images by using an image editing program of your choice. (See notebook on affine transformations.) Now, the goal is to align the pieces **automatically** using a smart algorithm.

Before scrolling down, take a moment to apply your image processing knowledge and develop a strategy: Which approach would you choose to compute the affine transformation needed to fit the puzzle piece into the overall arrangement?

In [ ]:
######################
###    Solution    ###
######################

# There are various ways to solve the puzzle. In this tutorial, we focus on
# aligning the pieces based on their shape. First, we extract the relevant
# contours from imageA and imageB. Then, we represent each contour as a 
# sequence of points. Finally, we align these point sequences by solving a 
# minimization problem: we determine the parameters of the affine transformation 
# (rotation, translation, scaling) that minimize the total distance between 
# corresponding points in the two sequences after transformation.

---

## **Preparations**

The usual preparations... The package `isp` provides some helper functions to easily render images in this Jupyter notebook.

In [ ]:
import sys
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Enable vectorized output (for nicer plots)
%config InlineBackend.figure_formats = ["svg"]

# Inline backend configuration
%matplotlib inline

# Functionality related to this course
sys.path.append("..")
import isp

# Jupyter / IPython configuration:
# Automatically reload modules when modified, if the extension is available
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

Let's load the images and take a closer look. Our goal is to overlay `imageB` onto `imageA`. To achieve this correctly, we need to preserve the *alpha channel* of the original images.

In [ ]:
# Read the image (flag IMREAD_UNCHANGED preserves the alpha channel)
imageA = cv.imread(filename="../data/images/water2/missing.jpg", 
                   flags=cv.IMREAD_COLOR)
imageB = cv.imread(filename="../data/images/water2/piece-scaled.jpg", 
                   flags=cv.IMREAD_COLOR)

# Convert the image from BGR to RGB
imageA = cv.cvtColor(imageA, cv.COLOR_BGR2RGB)
imageB = cv.cvtColor(imageB, cv.COLOR_BGR2RGB)

# Display the images side-by-side
titleA = f"Image A"
titleB = f"Image B"
isp.show_image_pair(imageA, imageB, title1=titleA, title2=titleB)

---

## **☆ Exercise 2: Binarize the image**

In our approach to solve the alignment problem, we focus on the shape of the puzzle piece. Our image processing toolbox offers us a tool to extract shapes: [findContours()](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html#gae4156f04053c44f886e387cff0ef6e08). As you've learned, contours can only be extracted from binary images.

**Tasks**:  
1. Binarize imageB so that the puzzle piece appears in white.  
2. Binarize imageA so that the cut-out region (the missing puzzle piece) appears in white.


In [ ]:
######################
###    EXERCISE    ###
######################

# imageA:
grayA = ...
binaryA = ...

isp.show_image_chain([imageA, grayA, binaryA], 
                     titles=["orig", "gray", "binary"])

# imageB:
grayB = ... 
binaryB = ...

isp.show_image_chain([imageB, grayB, binaryB], 
                     titles=["orig", "gray", "binary"])

In [ ]:
######################
###    SOLUTION    ###
######################

grayA = cv.cvtColor(imageA, cv.COLOR_RGB2GRAY)
ret, binaryA = cv.threshold(grayA, 0, 255, 
                            cv.THRESH_BINARY_INV+cv.THRESH_OTSU)

isp.show_image_chain([imageA, grayA, binaryA], 
                     titles=["orig", "gray", "binary"]);


grayB = cv.cvtColor(imageB, cv.COLOR_RGB2GRAY)
ret, binaryB = cv.threshold(grayB, 0, 255, 
                            cv.THRESH_BINARY_INV+cv.THRESH_OTSU)

isp.show_image_chain([imageB, grayB, binaryB], 
                     titles=["orig", "gray", "binary"])

---

## **☆ Exercise 3: Extract contours***

**Task**:
1. Extract the contour of binaryA
2. Extract the contour of binaryB

Hint: Use the morphological operation "closing" to reduce noise in the binary image.  
The corresponding OpenCV function is [`cv.morphologyEx()`](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html#ga67493776e3ad1a3df63883829375201f).


In [ ]:
######################
###    EXERCISE    ###
######################

# imageA:
contourA = ... 

# imageB:
contourB = ...

In [ ]:
######################
###    SOLUTION    ###
######################

imageAC = imageA.copy()
kernel = np.ones((3,3),np.uint8)
closedA = cv.morphologyEx(binaryA, cv.MORPH_CLOSE, kernel, iterations = 5)
contourA, hierarchy = cv.findContours(closedA, cv.RETR_TREE, cv.CHAIN_APPROX_NONE)
# Sort contours by size
contourA = sorted(contourA, key=cv.contourArea, reverse=True)
#contourB_sorted = sorted(contourB, key=cv.contourArea, reverse=True)

cv.drawContours(imageAC, contourA, 1, (255, 0, 0, 255), 6)
isp.show_image_chain([binaryA, closedA, imageAC], 
                     titles=["binary", "closed", "orig + contour"])

imageBC = imageB.copy()
kernel = np.ones((3,3), np.uint8)
closedB = cv.morphologyEx(binaryB, cv.MORPH_CLOSE, kernel, iterations = 4)
contourB, hierarchy = cv.findContours(closedB, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

imageBC = cv.drawContours(imageBC, contourB, -1, (255, 0, 0, 255), 3)
isp.show_image_chain([binaryB, closedB, imageBC], 
                     titles=["binary", "closed", "orig + contour"])

# findContour returns a list of contours, each contour is a list of points
# The inner contour is the second contour in the list.
contourA = contourA[1]
contourB = contourB[0]

---

## **☆ Exercise 4: Align images**

This task is a bit tricky - you don't need to understand every detail of the method.  
However, you’re encouraged to take a look at the alignment algorithm `align_points()` (see [../isp/geometry.py])

The algorithm essentially performs the following steps:
- Extract the contour points
- Preprocess the contour points (normalize and resample the point sequences)  
- Start with initial guesses for the transformation parameters (scale, rotation, translation)  
- Apply the current affine transformation to map all points from image B onto image A  
- For each transformed point from B, compute the Euclidean distance to the closest point in A  
- Use the sum of squared distances as the loss function to minimize  
- Apply an optimization method (e.g., gradient descent) to iteratively update the transformation parameters  
- Repeat the process for multiple initializations to reduce the risk of getting trapped in local minima


In [ ]:
######################
#   DRAWING TOOL     #
######################

def overlay_images_rgba(imageA, imageB):
    """ Overlays imageB on top of imageA. The alpha channel of imageB
    is used to determine which part of imageB is visible. The result is
    a new image with the same size as imageA.
    """
    result = imageA.copy()
    mask = imageB[:,:,3] > 0
    result[mask] = imageB[mask]
    return result


def overlay_images(imageA, imageB, maskB):
    """ Overlays imageB on top of imageA. The mask is used to determine
    which part of imageB is visible. The result is a new image with the
    same size as imageA.
    """
    result = imageA.copy()
    result[maskB] = imageB[maskB]
    return result

In [ ]:
from isp.geometry import align_points

# Extract the points from the contours as a list of 2D points
# (OpenCV uses a special convention how the contour points are stored)
pointsA = contourA[:,0,:]
pointsB = contourB[:,0,:]
print("Contour pointsA:", pointsA.shape)
print(pointsA[:5], end="\n ...\n\n")
print("Contour pointsB:", pointsB.shape)
print(pointsB[:5], end="\n ...\n\n")

# Number of points to resample. This affects alignment quality:
# more points typically lead to higher accuracy, but also increase
# computation time.
n = 1000

# Here's the key function that performs the alignment!
# Curious minds are encouraged to explore its source code.
# Note that the algorithm runs multiple alignment attempts —
# since optimization problems like this can get stuck in local minima,
# repeating the process increases the chance of finding a global minimum.
# The parameter *n_attempts* controls how many attempts are made.
trafo = align_points(data_a = pointsA, data_b = pointsB, 
                     n_samples=n, n_attemps=6, 
                     with_scale=True)

In [ ]:
# Let's apply the transformation to the second image. Since we 
# want to overlay it onto imageA, we set the output size to match 
# the dimensions of imageA.
imageB_aligned = cv.warpAffine(imageB, trafo, imageA.shape[:2])
binaryB_aligned = cv.warpAffine(binaryB, trafo, imageA.shape[:2]) > 128
imageB_overlay = overlay_images(imageA, imageB_aligned, binaryB_aligned)
isp.show_image_chain([imageA, imageB_aligned, imageB_overlay],
                     titles=["imageA: Target", "Image B: Single piece", "Solution"],
                     shape=None)

## **☆ Exercise 5: Wrap up**

Tataaaa. We have solved the puzzle! 

**Task**: Summarize the steps required to complete this task in your own words. Would you be able to solve the puzzle if it were presented to you in this form again?

